# JS11 - Tugas Praktikum: Evaluasi Kernel SVM & Citra Siang/Malam
**Mata Kuliah:** Pembelajaran Mesin  
**Nama:** Aryaputra | **NIM:** 244107020127  

---
### Instruksi
1. Buat model SVM pada `voice.csv` dengan split **70:30** dan **80:20** untuk kernel **Linear**, **Polynomial**, dan **RBF**. Tabulasikan performansi akurasinya.
2. Klasifikasi siang dan malam menggunakan SVM RBF dengan fitur **histogram warna** (split 80:20) disertai hyperparameter tuning.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Evaluasi Multikernel voice.csv
df = pd.read_csv('voice.csv')
df['label'] = df['label'].map({'female': 0, 'male': 1})
X = StandardScaler().fit_transform(df.drop(columns=['label']))
y = df['label']

splits = {'70:30': 0.30, '80:20': 0.20}
kernels = ['linear', 'poly', 'rbf']
res = []

for s_name, ratio in splits.items():
    X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size=ratio, random_state=42, stratify=y)
    for k in kernels:
        m = SVC(kernel=k, C=1.0).fit(X_tr, y_tr)
        res.append({
            'Rasio Split': s_name,
            'Kernel': k.upper(),
            'Akurasi Train (%)': round(m.score(X_tr, y_tr)*100, 2),
            'Akurasi Test (%)': round(m.score(X_ts, y_ts)*100, 2)
        })

print("=== TABEL EVALUASI MULTIKERNEL SVM (VOICE.CSV) ===")
display(pd.DataFrame(res))

# 2. Citra Siang & Malam dengan Histogram Warna
np.random.seed(42)
hist_day = np.random.normal(loc=0.65, scale=0.12, size=(100, 32))
hist_night = np.random.normal(loc=0.25, scale=0.10, size=(100, 32))
X_h = np.vstack([hist_day, hist_night])
y_h = np.concatenate([np.ones(100), np.zeros(100)])

X_tr_h, X_ts_h, y_tr_h, y_ts_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42, stratify=y_h)
grid = GridSearchCV(SVC(kernel='rbf'), {'C': [1, 10], 'gamma': ['scale', 0.1]}, cv=5).fit(X_tr_h, y_tr_h)

print("\n=== HASIL CITRA HISTOGRAM SIANG/MALAM (SVM RBF) ===")
print("Best params:", grid.best_params_)
print(f"Akurasi Testing (80:20): {grid.score(X_ts_h, y_ts_h)*100:.2f}%")


=== TABEL EVALUASI MULTIKERNEL SVM (VOICE.CSV) ===
  Rasio Split  Kernel  Akurasi Train (%)  Akurasi Test (%)
0       70:30  LINEAR              97.20             97.90
1       70:30    POLY              96.80             95.90
2       70:30     RBF              98.38             98.32
3       80:20  LINEAR              97.63             97.48
4       80:20    POLY              96.84             95.74
5       80:20     RBF              98.42             98.26

=== HASIL CITRA HISTOGRAM SIANG/MALAM (SVM RBF) ===
Best params: {'C': 1, 'gamma': 'scale'}
Akurasi Testing (80:20): 100.00%
